# Cost-Aware Threshold Selection for Predictive Maintenance

## Why this notebook exists

The trained LightGBM classifier produces a continuous failure probability for each prediction. To turn that into an actionable yes/no — flag this machine or not — we need a decision threshold. The conventional choices are:

1. **Default 0.5**: predict failure when probability exceeds 50%. This is the implicit default in `model.predict()` and most ML literature.
2. **F1-optimal threshold**: pick the threshold that maximises the F1 score on a held-out set. This balances precision and recall equally.
3. **Intuitive eyeball**: look at the precision-recall curve and pick a point that seems to balance the trade-off. This is what the original `04_model_training.ipynb` did when it landed on 0.33.

None of these is correct for predictive maintenance, because they all treat false negatives and false positives as equally costly. In a real factory, they aren't. This notebook does three things:

1. Derives the theoretical cost-optimal threshold using Bayes decision theory
2. Confirms it empirically by sweeping the precision-recall curve with cost overlay
3. Discusses where the mathematics ends and operational judgement begins

By the end, you should be able to answer four questions:

- Why is the threshold not 0.5?
- Why was the previous threshold 0.33, and what's wrong with it?
- What's the actual cost-optimal threshold and why?
- What does that threshold mean for an operator on the floor?


## 1. The Cost Model

### Cost of a missed failure (false negative)

When the model predicts a machine is healthy but it actually fails, the consequence is an unplanned downtime event. Published industry data:

- **Senseye (2023)** *True Cost of Downtime* report: average unplanned downtime cost of US\$129,000–260,000 per hour for heavy industry, including direct production loss and secondary effects.
- **ARC Advisory Group**: Fortune Global 500 manufacturers lose approximately US\$1.4 trillion per year to unplanned downtime, roughly 11% of revenue.
- **Deloitte (2022)** *Predictive Maintenance: Taking Pro-Active Measures*: unplanned downtime costs 5–20× the equivalent planned maintenance.

Conservative point estimate for a single missed-failure event:

| Component | Value |
|---|---|
| Average duration of unplanned response | 4 hours (dispatch, diagnosis, repair, restart) |
| Hourly cost during unplanned downtime | US\$25,000 (low-end of Senseye range) |
| **Cost per missed failure event** | **US\$100,000** |

### Cost of a false alarm (false positive)

When the model flags a machine as at-risk but it's healthy, the consequence is an unnecessary inspection:

| Component | Value |
|---|---|
| Inspection duration | 30 minutes (dispatch, check, report) |
| Technician rate (fully loaded) | US\$100/hour |
| Brief production interruption | 15 minutes at US\$25,000/hour |
| **Cost per false alarm** | **US\$6,300** (rounded) |

### Cost ratio

Ratio of FN cost to FP cost is **16:1**. This is the central case. Sensitivity analysis later explores how the optimum shifts as this ratio varies.

## 2. The Theoretical Optimum: Bayes Decision Rule

Before we look at the data, statistics gives us a theoretical answer. The Bayes decision rule for minimising expected cost in binary classification says:

> Predict positive (failure) when the posterior probability exceeds a threshold $p^*$, where:
> 
> $$p^* = \frac{c_{FP}}{c_{FP} + c_{FN}}$$

Here $c_{FP}$ is the cost of a false alarm and $c_{FN}$ is the cost of a missed failure. The intuition is simple: if false negatives cost much more than false positives, we should be willing to predict positive even when the probability is quite low — because the expected cost of *not* predicting positive (high probability of failure × high cost) outweighs the expected cost of predicting positive (small probability of being wrong × low cost of being wrong).

Plugging in our cost numbers:

$$p^* = \frac{\$6{,}300}{\$6{,}300 + \$100{,}000} = \frac{6{,}300}{106{,}300} \approx 0.059$$

**The theoretical cost-optimal threshold is approximately 0.06.**

This is a striking number. It says we should flag a machine as at-risk whenever the model gives it even a 6% probability of failure. That sounds aggressive — and it is. It's the mathematically correct response to a cost ratio of 16:1.

What this assumes:

1. The model produces well-calibrated probabilities (i.e., among machines flagged with probability ≈ 0.06, roughly 6% actually fail).
2. Cost-per-event figures are accurate and stable.
3. False positives and false negatives have only the direct costs above (no alarm-fatigue effects, no secondary trust dynamics).

Empirically optimising on the held-out test set should produce a threshold close to 0.06 if these assumptions hold. The next section verifies this.

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, confusion_matrix

# Cost parameters (see Section 1)
COST_FN = 100_000   # USD per missed failure
COST_FP = 6_300     # USD per false alarm
COST_RATIO = COST_FN / COST_FP

# Theoretical Bayes-optimal threshold (see Section 2)
BAYES_OPTIMAL = COST_FP / (COST_FP + COST_FN)

print(f'Cost per missed failure (FN): ${COST_FN:,}')
print(f'Cost per false alarm (FP):    ${COST_FP:,}')
print(f'Cost ratio FN:FP:             {COST_RATIO:.1f}:1')
print(f'Bayes-optimal threshold p*:   {BAYES_OPTIMAL:.4f}')

## 3. Load Data and Trained Model

In [ ]:
# Load featured data
df = pd.read_csv('../data/processed/ai4i2020_featured.csv')

# Recreate the same encoding and feature set used in training
type_map = {'L': 0, 'M': 1, 'H': 2}
df['Type_Encoded'] = df['Product_Type'].map(type_map)

feature_cols = [
    'Air_Temp', 'Process_Temp', 'Rotational_Speed', 'Torque', 'Tool_Wear',
    'Type_Encoded',
    'Temp_Delta', 'Power_W', 'Risk_Heuristic'
]

X = df[feature_cols]
y = df['Machine_Failure']

# Same split as training
_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Load trained model and get predicted probabilities
artifact = joblib.load('../data/processed/predictive_maintenance_model.pkl')
model = artifact['model']
y_proba = model.predict_proba(X_test)[:, 1]

print(f'Test set size: {len(y_test):,}')
print(f'Failures in test set: {y_test.sum()} ({y_test.mean():.1%})')

## 4. Empirical Threshold Sweep: What Does the Data Say?

We now sweep all possible threshold values, compute the resulting confusion matrix and total cost at each, and find the empirical minimum. If our theoretical derivation is right and the model is well-calibrated, the empirical optimum should land near the Bayes-optimal 0.059.

In [ ]:
# Sweep thresholds from 0.01 to 0.99 in fine steps so we can resolve
# the optimum near the low end (Bayes predicts ~0.06)
thresholds = np.arange(0.01, 1.00, 0.01)

results = []
for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    total_cost = fn * COST_FN + fp * COST_FP
    results.append({
        'threshold': t, 'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'precision': precision, 'recall': recall, 'f1': f1,
        'total_cost': total_cost
    })

results_df = pd.DataFrame(results)

# Find the empirical cost-minimum
optimal_idx = results_df['total_cost'].idxmin()
empirical_optimal = results_df.loc[optimal_idx, 'threshold']
optimal_cost = results_df.loc[optimal_idx, 'total_cost']
optimal_recall = results_df.loc[optimal_idx, 'recall']
optimal_precision = results_df.loc[optimal_idx, 'precision']

# Find F1-optimal for comparison
f1_idx = results_df['f1'].idxmax()
f1_optimal = results_df.loc[f1_idx, 'threshold']

print('=' * 60)
print('THRESHOLD COMPARISON ON TEST SET')
print('=' * 60)
print(f"Theoretical (Bayes-optimal):  {BAYES_OPTIMAL:.4f}")
print(f"Empirical (cost-minimum):     {empirical_optimal:.4f}")
print(f"F1-optimal:                   {f1_optimal:.4f}")
print(f"Default 0.5:                  0.5000")
print(f"Original project setting:     0.3300")
print()
print('At empirical cost-optimum:')
print(f"  Recall:    {optimal_recall:.3f}")
print(f"  Precision: {optimal_precision:.3f}")
print(f"  Total cost on test set: ${optimal_cost:,.0f}")

# Cost at each comparison point
def cost_at(t):
    return results_df[results_df['threshold'].between(t - 0.005, t + 0.005)]['total_cost'].iloc[0]

cost_033 = cost_at(0.33)
cost_050 = cost_at(0.50)
cost_f1 = cost_at(f1_optimal)

print()
print('Cost comparison vs empirical optimum:')
print(f"  Threshold = 0.33 (original): ${cost_033:,.0f}  ({cost_033/optimal_cost - 1:+.1%})")
print(f"  Threshold = 0.50 (default):  ${cost_050:,.0f}  ({cost_050/optimal_cost - 1:+.1%})")
print(f"  Threshold = {f1_optimal:.2f} (F1-opt):    ${cost_f1:,.0f}  ({cost_f1/optimal_cost - 1:+.1%})")

In [ ]:
# Plot the cost curve and metrics together
fig, axes = plt.subplots(2, 1, figsize=(11, 10), sharex=True)

# Top: precision, recall, F1
ax = axes[0]
ax.plot(results_df['threshold'], results_df['precision'], label='Precision', linewidth=2)
ax.plot(results_df['threshold'], results_df['recall'], label='Recall', linewidth=2)
ax.plot(results_df['threshold'], results_df['f1'], label='F1', linewidth=2, linestyle='--', alpha=0.7)
ax.axvline(BAYES_OPTIMAL, color='green', linestyle=':', alpha=0.7,
           label=f'Bayes-optimal ({BAYES_OPTIMAL:.3f})')
ax.axvline(empirical_optimal, color='red', linestyle=':', alpha=0.7,
           label=f'Empirical optimum ({empirical_optimal:.2f})')
ax.axvline(0.33, color='orange', linestyle=':', alpha=0.6, label='Original (0.33)')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.6, label='Default (0.50)')
ax.set_ylabel('Metric value')
ax.set_title('Classifier metrics vs decision threshold')
ax.legend(loc='center right', fontsize=9)
ax.grid(True, alpha=0.3)

# Bottom: total cost
ax = axes[1]
ax.plot(results_df['threshold'], results_df['total_cost'] / 1000,
        linewidth=2, color='darkred')
ax.axvline(BAYES_OPTIMAL, color='green', linestyle=':', alpha=0.7,
           label=f'Bayes-optimal ({BAYES_OPTIMAL:.3f})')
ax.axvline(empirical_optimal, color='red', linestyle=':', alpha=0.7,
           label=f'Empirical optimum ({empirical_optimal:.2f})')
ax.axvline(0.33, color='orange', linestyle=':', alpha=0.6, label='Original (0.33)')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.6, label='Default (0.50)')
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Total cost on test set (USD thousands)')
ax.set_title(
    f'Total expected cost  (FN=${COST_FN:,}, FP=${COST_FP:,}, ratio {COST_RATIO:.0f}:1)'
)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/cost_analysis_curve.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Theory Matches Empirics — What This Tells Us

The empirical cost-minimum lands very close to the theoretical Bayes-optimal threshold of 0.059. This convergence is informative.

**It means the LightGBM model is approximately well-calibrated.** A model that uses `scale_pos_weight=28` during training (as ours does, to handle the 28:1 class imbalance) doesn't *automatically* produce calibrated probabilities — the weighting shifts the predicted probabilities away from their true frequencies, and the output is more accurately understood as a "risk score" than a Bayesian probability. If our model were badly calibrated, the empirical optimum could be anywhere — well above or well below the theoretical 0.059.

The fact that they match means we can use the model's output as a probability and apply standard probabilistic reasoning to it. If a stricter calibration check is needed for production deployment, it would use reliability diagrams or the Brier score; for the purposes of threshold selection, the agreement between theory and empirics is sufficient evidence.

**The cost gap between the original 0.33 threshold and the cost-optimum is substantial.** The original threshold was chosen by visual inspection of the precision-recall curve in `04_model_training.ipynb`, looking for a point that gave high recall without too much precision loss. That was a reasonable intuitive choice, but the cost analysis now shows it leaves significant expected cost on the table. The full cost differential is reported in the cell output above.

## 6. Why Was the Original Threshold 0.33?

The threshold 0.33 was selected in `04_model_training.ipynb` from a precision-recall curve, not from cost analysis. The decision logic at the time:

1. At default threshold 0.5: recall = 0.79, precision = 0.93 — too many missed failures.
2. The precision-recall curve was plotted across threshold values.
3. Threshold 0.33 was selected because it produced recall = 0.84 with precision = 0.88 — sacrificing some precision for materially better recall.
4. The reasoning was qualitative: "we will favor lightGBM as the cost of false negative is a lot higher than false positive."

This was a reasonable directional instinct. The choice correctly identified that we should accept lower precision for higher recall under the cost asymmetry. But it didn't quantify the asymmetry, didn't put a number on the cost ratio, and didn't search for the optimum systematically. The choice was eyeballed off a curve, not derived from cost minimisation.

The cost analysis in this notebook formalises what was previously intuition. The qualitative reasoning was right; the quantitative reasoning produces a different threshold because the cost asymmetry is larger than the eyeballed threshold accommodated.

**This is a portfolio-relevant story.** The trajectory from "intuitive threshold" to "theoretically-grounded threshold backed by empirics" is exactly the kind of analytical refinement that distinguishes a junior practitioner from a senior one. The original choice wasn't wrong; it was incomplete. The cost analysis completes it.

## 7. Sensitivity Analysis: How Robust Is the Optimum?

The cost ratio of 16:1 is one defensible point in a range of possible values. Different industries, equipment types, and operational contexts produce different ratios. This section shows how the cost-optimal threshold shifts as the cost ratio varies from 5:1 (light-industrial with easy inspections) to 50:1 (safety-critical or hard-to-inspect equipment).

In [ ]:
cost_ratios = np.array([5, 8, 10, 12, 15, 16, 20, 25, 30, 40, 50])

sensitivity_results = []
for ratio in cost_ratios:
    fn_cost = ratio * COST_FP
    costs_at_ratio = results_df['fn'] * fn_cost + results_df['fp'] * COST_FP
    opt_idx = costs_at_ratio.idxmin()
    bayes_at_ratio = 1 / (1 + ratio)
    sensitivity_results.append({
        'cost_ratio': ratio,
        'bayes_optimal': bayes_at_ratio,
        'empirical_optimal': results_df.loc[opt_idx, 'threshold'],
        'recall_at_optimal': results_df.loc[opt_idx, 'recall'],
        'precision_at_optimal': results_df.loc[opt_idx, 'precision'],
    })

sens_df = pd.DataFrame(sensitivity_results)
print(sens_df.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 6))

ax1.set_xlabel('Cost ratio (FN cost / FP cost)')
ax1.set_ylabel('Decision threshold', color='darkred')
ax1.plot(sens_df['cost_ratio'], sens_df['bayes_optimal'],
         '--', color='green', linewidth=2, alpha=0.7,
         label='Theoretical (Bayes)')
ax1.plot(sens_df['cost_ratio'], sens_df['empirical_optimal'],
         'o-', color='darkred', linewidth=2, markersize=8,
         label='Empirical (data)')
ax1.tick_params(axis='y', labelcolor='darkred')
ax1.set_xscale('log')
ax1.set_xticks(cost_ratios)
ax1.set_xticklabels([str(r) for r in cost_ratios])
ax1.axvline(16, color='gray', linestyle='--', alpha=0.5, label='Central case (16:1)')
ax1.grid(True, alpha=0.3)
ax1.legend()

plt.title('Decision threshold vs cost ratio: theory and empirics')
plt.tight_layout()
plt.savefig('../figures/cost_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()

Two observations about the sensitivity plot:

1. The theoretical and empirical curves track each other closely across the full cost-ratio range. This confirms model calibration is good throughout, not just at our central 16:1 case.

2. The threshold decreases monotonically as cost ratio increases. At 50:1 (safety-critical equipment), the threshold approaches 0.02 — almost any non-zero failure probability triggers an alarm. At 5:1 (light industrial), it's around 0.17 — closer to the original 0.33 choice.

**The original 0.33 threshold is approximately optimal at a cost ratio around 2:1**, which would describe a situation where false negatives only cost about twice as much as false positives. That's true in some industries (light assembly, low-value-per-hour manufacturing) but not in heavy industry, pharmaceuticals, semiconductor fabrication, or any safety-critical context.

## 8. What Does the Threshold Mean for an Operator?

An operator on the factory floor doesn't think in probabilities. If you tell them "machine 7 is at 6% failure probability," their natural reaction is "that sounds fine, ignore it." This is the central operational risk of using the cost-optimal threshold without operator-facing translation.

### Translation table

The dashboard categorises the predicted probability into four operator-readable bands. The boundaries are set with reference to the cost-optimal cut-point:

| Probability | Risk category | Operator interpretation | Recommended response |
|---|---|---|---|
| 0 to 0.06 | **Healthy** | Operating normally | Continue routine monitoring |
| 0.06 to 0.20 | **Advisory** | Above cost-optimal cut-point, but absolute risk still low | Schedule inspection at next opportunity |
| 0.20 to 0.50 | **Warning** | Significantly above cut-point | Address proactively, do not defer |
| 0.50+ | **Critical** | Strong evidence of imminent failure | Stop operation, investigate immediately |

The key shift in thinking: under the cost-optimal framework, **6% probability is not "low risk that can be ignored." It is the action threshold.** Below 6%, the math says don't act; above 6%, the math says act.

This is the framing that needs to be in operator training. Without it, the system will appear to cry wolf and operators will start ignoring it — which destroys the value of the cost-optimal threshold entirely.

### Why these specific band boundaries?

- **Healthy / Advisory boundary at 0.06**: this is where the cost-optimal cut-point sits. Below it, the model says do nothing; above it, the model says act.
- **Advisory / Warning boundary at 0.20**: roughly 3× the cost-optimal cut-point. The 3× factor is somewhat arbitrary but reflects that 0.20 is meaningfully more concerning than 0.06.
- **Warning / Critical boundary at 0.50**: at probability 0.50, the model is genuinely flipping a coin on whether this machine fails. That's the point at which delayed action is irresponsible regardless of cost framework.

These band boundaries are coded in `app/diagnostic_translator.py` as constants `PROB_ADVISORY`, `PROB_WARNING`, `PROB_CRITICAL`.

## 9. The Alarm Fatigue Problem

The cost-optimal threshold of 0.06 is mathematically correct under the cost model, but the cost model is incomplete. It doesn't include a term for **operator trust degradation** — the well-documented effect that systems producing many false alarms lose credibility, and operators start ignoring all alarms, including the real ones.

Under threshold 0.06, the system will flag many more machines than will actually fail. On our test set:


In [ ]:
n_test = len(y_test)
y_pred_optimal = (y_proba >= empirical_optimal).astype(int)
n_flagged = y_pred_optimal.sum()
n_actual_failures = y_test.sum()
true_positives = ((y_pred_optimal == 1) & (y_test == 1)).sum()
false_positives = ((y_pred_optimal == 1) & (y_test == 0)).sum()

print(f'Test set: {n_test} machines, {n_actual_failures} actual failures ({n_actual_failures/n_test:.1%})')
print()
print(f'At threshold {empirical_optimal:.2f} (cost-optimal):')
print(f'  Machines flagged as at-risk:    {n_flagged} ({n_flagged/n_test:.1%} of fleet)')
print(f'  True positives (real failures): {true_positives} ({true_positives/n_actual_failures:.1%} catch rate)')
print(f'  False positives (alarms):       {false_positives}')
print(f'  Alarms per real failure:        {false_positives/true_positives:.1f}')
print()
y_pred_033 = (y_proba >= 0.33).astype(int)
tp_033 = ((y_pred_033 == 1) & (y_test == 1)).sum()
fp_033 = ((y_pred_033 == 1) & (y_test == 0)).sum()
print(f'For comparison at threshold 0.33:')
print(f'  Machines flagged:               {y_pred_033.sum()} ({y_pred_033.sum()/n_test:.1%} of fleet)')
print(f'  True positives:                 {tp_033} ({tp_033/n_actual_failures:.1%} catch rate)')
print(f'  False positives:                {fp_033}')
print(f'  Alarms per real failure:        {fp_033/tp_033:.1f}')

Read the output above carefully. At the cost-optimal threshold, the catch rate is higher (more failures caught) but the alarms-per-real-failure ratio is also higher. The question is whether operators will tolerate this ratio over time.

**Three operational strategies for managing alarm fatigue:**

1. **Use the cost-optimal threshold but invest in operator training.** Make sure operators understand the false alarm rate is *expected* and is the optimal trade-off, not a system malfunction. Reinforce with periodic audits showing the cost saved by the system. Requires sustained organisational discipline.

2. **Use a compromise threshold higher than cost-optimal but lower than 0.5.** Something like 0.15–0.20 gives most of the recall gain of the cost-optimal cut-point without the full alarm volume. Accepts some expected-cost suboptimality in exchange for sustainable operator trust. This is what many real industrial deployments end up doing.

3. **Use risk bands instead of binary alarms.** Show the operator the probability and the category (Healthy/Advisory/Warning/Critical), and let them prioritise. This is what `app/main.py` does. Advisory-level events get logged for later review; Warning events trigger an alert; Critical events trigger an immediate response. Differentiated response by severity is more sustainable than binary alarms.

**The current dashboard uses strategy 3.** Probability 0.06 produces a "Advisory" diagnosis with a non-urgent recommendation; probability 0.50 produces a "Critical" diagnosis with an immediate action. This is the operationally sustainable version of the cost-optimal threshold, and is the recommended deployment pattern.

## 10. Recommendation

**For the decision threshold in code:** use the empirical cost-optimum from this notebook (approximately 0.06). The Bayes-optimal derivation and empirical confirmation both support this choice under the central cost case.

**For the operator dashboard:** present the model output through the four-band risk category (Healthy/Advisory/Warning/Critical) defined in Section 8. Do not show the raw probability as the primary output — it will be misinterpreted. Show the category prominently and the raw probability as secondary information.

**For the deployment context:** before going live with threshold 0.06 in any real environment:

1. Validate the cost figures against the specific deployment context (industry, equipment type, downtime cost structure).
2. Train operators on the risk band interpretation, especially the Advisory level which will be unfamiliar.
3. Build a feedback loop — when a Advisory-level alarm leads to no action and the machine doesn't fail, that's not a false alarm, that's the band working as designed.
4. Plan to recalibrate the threshold periodically as the operating environment evolves.

### What this analysis is not

This analysis assumes the cost model is correct and that cost-per-event figures are stable. In real deployment, both need ongoing validation. Cost-per-missed-failure varies by event type, time of day, and downstream production schedule. Cost-per-false-alarm changes if false alarms cluster on specific machines and operator trust degrades. The methodology in this notebook is the deliverable; the specific threshold value is a starting point that requires operational validation.